# P.R.I.S.M. Fine-Tuning Experiment 2: The Clean Stack (rsLoRA + Triton MoE + MXFP4)

## Overview

This notebook implements an ablation study for the P.R.I.S.M. architecture, removing conflicting optimization techniques to stabilize gradients and prevent "quantization amnesia" of custom special tokens:

- **rsLoRA (Rank-Stabilized LoRA):** Maintains gradient stability at high rank (r=64) for MoE models without causing oversized updates.
- **Standard 8-bit AdamW:** Replaces LoRA+ to prevent overwriting rigid formatting templates.
- **Unsloth Triton MoE Kernels:** Leverages `torch._grouped_mm` for memory savings (kept fully compatible by disabling DoRA).
- **I-Matrix Calibration:** Protects special `<unusedX>` tokens during quantization by heavily weighting their activation pathways.

## Production Target

**Quantization Format:** MXFP4 with I-Matrix calibration
**Reason:** MXFP4 provides an optimal balance of quality and size for production deployment, and the I-Matrix ensures the strict structural formatting pathways are preserved.

In [ ]:
# @title 1. Install Dependencies (Bleeding Edge)

# Upgrade PyTorch with CUDA 12.8 support
# !pip install --upgrade --force-reinstall torchvision --index-url https://download.pytorch.org/whl/cu128

# This command uses Unsloth's optimized installation path for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-31kkenr9/unsloth_3b64ebfaf35a48d2a5b5a6bed74db2ee
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-31kkenr9/unsloth_3b64ebfaf35a48d2a5b5a6bed74db2ee
  Resolved https://github.com/unslothai/unsloth.git to commit d1f9ab659fc5d3309e9e40166be309369bedf852
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 182.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 179.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 26.1 MB/s eta 0:00:0

In [ ]:
# Upgrading specific Hugging Face libraries to ensure compatibility
!pip install --upgrade --no-cache-dir trl transformers

print("✅ Additional dependencies installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 300.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 490.1 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which is i

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# @title 2. Environment Setup

import os
import torch
from google.colab import drive, userdata

# Define project directory
project_dir = "/content/drive/MyDrive/PRISM_FineTuning/"
os.makedirs(project_dir, exist_ok=True)

# Enable ModelScope for HuggingFace timeout workaround
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

# Enable Triton MoE kernels (critical for Gemma 4)
os.environ['UNSLOTH_TRITON_MOE'] = '1'

# Enable grouped_mm for memory savings
os.environ['UNSLOTH_GROUPED_MM'] = '1'

print(f"✅ Project directory: {project_dir}")
print(f"✅ Triton MoE enabled: {os.environ.get('UNSLOTH_TRITON_MOE')}")
print(f"✅ Grouped MM enabled: {os.environ.get('UNSLOTH_GROUPED_MM')}")

✅ Project directory: /content/drive/MyDrive/PRISM_FineTuning/
✅ Triton MoE enabled: 1
✅ Grouped MM enabled: 1


In [ ]:
!pip install modelscope -U
# Install Rust-based fast transfer library
!pip install -U hf-transfer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 114.3 MB/s eta 0:00:00


In [ ]:
# @title 3. Load Base Model with Unsloth
import os
import unsloth
from unsloth import FastModel

# Enable Rust-based fast downloads
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Model configuration
model_name = "google/gemma-4-26b-a4b-it"
hf_token = userdata.get("HF_TOKEN")

# # Define cache directory on Google Drive
# cache_dir = "/content/drive/MyDrive/PRISM_FineTuning/model_cache"
# os.makedirs(cache_dir, exist_ok=True)

print(f"Loading {model_name}...")
# print(f"Caching model to: {cache_dir}")

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 8192,  # Full context window
    load_in_4bit = True,
    token = hf_token,
    # cache_dir = cache_dir,
)

print(f"✅ Model loaded: {model_name}")
print(f"✅ Max sequence length: 8192")
print(f"✅ Parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading google/gemma-4-26b-a4b-it...


2026-05-08 08:59:53,637 - modelscope - INFO - Got 35 files, start to download ...


Processing 35 items:   0%|          | 0.00/35.0 [00:00<?, ?it/s]

2026-05-08 09:13:50,286 - modelscope - INFO - Finish downloading 35 files for repo 'unsloth/gemma-4-26b-a4b-it'


==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.8.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

✅ Model loaded: google/gemma-4-26b-a4b-it
✅ Max sequence length: 8192
✅ Parameters: 25,805,933,872


In [ ]:
# @title 4. Configure rsLoRA Adapters

from peft import LoraConfig, get_peft_model
import torch.nn as nn

print("Configuring rsLoRA Adapters...")

# rsLoRA configuration
lora_config = {
    "use_dora": False,  # Disabled to prevent grouped_mm coalescing issues
    "use_rslora": True,  # Rank-Stabilized LoRA for gradient stability
    "r": 64,
    "lora_alpha": 64,  # Alpha = R for rsLoRA standard practice
    "lora_dropout": 0,
    "bias": "none",
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "moe_gate",  # Critical for Gemma 4 MoE
        # "embed_tokens", "lm_head" # Use LoRA on embeddings instead of full-rank updates
    ],
}

# Apply configuration
model = FastModel.get_peft_model(
    model,
    r = lora_config["r"],
    target_modules = lora_config["target_modules"],
    lora_alpha = lora_config["lora_alpha"],
    lora_dropout = lora_config["lora_dropout"],
    use_dora = lora_config["use_dora"],
    use_rslora = lora_config["use_rslora"],
    bias = lora_config["bias"],
    init_lora_weights = True, # Standard initialization, no PiSSA
)

print("✅ rsLoRA configuration applied")
print(f"✅ Rank (r): {lora_config['r']}")
print(f"✅ Alpha: {lora_config['lora_alpha']}")

Configuring rsLoRA Adapters...
Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'moe_gate']. Enabling LoRA on MoE parameters: ['experts.gate_up_proj', 'experts.down_proj']
✅ rsLoRA configuration applied
✅ Rank (r): 64
✅ Alpha: 64


In [6]:
# @title 5. Register P.R.I.S.M. Special Tokens

# P.R.I.S.M. Token Map
PRISM_TOKENS = {
    "<|think|>": "<unused0>",
    "<|channel>thought": "<unused1>",
    "<|channel>": "<unused2>",
    "<|tool_call>": "<unused3>",
    '<"|>': "<unused4>",
    "[Logical Chain]": "<unused5>",
    "[Competing Hypotheses]": "<unused6>",
    "[Discarded Paths]": "<unused7>",
    "▶ Selected:": "<unused8>",
    "✗ Discarded:": "<unused9>",
}

# Add special tokens to tokenizer
special_tokens = list(PRISM_TOKENS.values())

# Get the actual tokenizer if it's wrapped in a processor
actual_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
actual_tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

# Note: We do NOT need to resize model embeddings because <unused0> through <unused99>
# are already present in the Gemma vocabulary!

# Ensure embeddings are trainable (critical for DoRA)
for name, param in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        param.requires_grad = True

print("✅ P.R.I.S.M. tokens registered")
print(f"✅ Tokenizer size: {len(actual_tokenizer)}")
print(f"✅ Model embedding size: {model.get_input_embeddings().weight.shape[0]}")

NameError: name 'tokenizer' is not defined

In [7]:
# @title 6. Load and Prepare Enhanced Dataset

import json
from datasets import Dataset

# Load original dataset
dataset_path = os.path.join(project_dir, "deliberation_dataset.json")

if os.path.exists(dataset_path):
    with open(dataset_path, "r") as f:
        original_data = json.load(f)
    print(f"✅ Loaded original dataset: {len(original_data)} examples")
else:
    print(f"⚠️ Dataset not found at {dataset_path}")
    original_data = []

# Load augmented safe combinations (if available)
safe_combinations_path = os.path.join(project_dir, "safe_combinations_augmented.json")

if os.path.exists(safe_combinations_path):
    with open(safe_combinations_path, "r") as f:
        safe_data = json.load(f)
    print(f"✅ Loaded safe combinations: {len(safe_data)} examples")
else:
    print("⚠️ Safe combinations not found, using original dataset only")
    safe_data = []

# Combine datasets
combined_data = original_data + safe_data
print(f"✅ Total examples: {len(combined_data)}")

# System prompt
system_prompt = """You are a clinical deliberation AI. You must rigidly format your output exactly according to the following schema. Do not deviate or add conversational filler.

EXPECTED OUTPUT SCHEMA:
<unused0>
<unused1>
<unused5>
[Numbered step-by-step logical chain]

<unused6>
[Enumerate competing interpretations with probability estimates, including supporting and weakening evidence]

<unused7>
<unused9> Discarded: [Explanation of discarded paths]

<unused8> Selected: [Final chosen interpretation]
<unused3> [Optional tool calls to verify claims]
<unused2>
[🔴, 🟡, or 🟢] [Clinical reasoning summary]

Confidence: ✅ [LEVEL]

Recommendation: [Actionable advice]"""

# Prompt template (Keep this exactly as you have it)
prompt_template = system_prompt + """
<bos><start_of_turn>user
{instruction}<end_of_turn>
<start_of_turn>model
<unused0>
<unused1>
{thought_process}
<unused2>
{output}<end_of_turn><eos>"""

def format_prompts(examples):
    instructions = examples["instruction"]
    thoughts = examples["thought_process"]
    outputs = examples["output"]
    texts = []

    for instruction, thought, output in zip(instructions, thoughts, outputs):
        # Map headers in thought process
        for human, model_token in PRISM_TOKENS.items():
            thought = thought.replace(human, model_token)

        text = prompt_template.format(
            instruction=instruction,
            thought_process=thought,
            output=output
        )
        texts.append(text)

    return {"text": texts}

# Convert to Hugging Face Dataset
data_dict = {
    "instruction": [item["instruction"] for item in combined_data],
    "thought_process": [item["thought_process"] for item in combined_data],
    "output": [item["output"] for item in combined_data]
}

deliberation_dataset = Dataset.from_dict(data_dict)
deliberation_dataset = deliberation_dataset.map(format_prompts, batched=True)

print(f"✅ Dataset prepared: {len(deliberation_dataset)} examples")

✅ Loaded original dataset: 1001 examples
✅ Loaded safe combinations: 10 examples
✅ Total examples: 1011


Map:   0%|          | 0/1011 [00:00<?, ? examples/s]

✅ Dataset prepared: 1011 examples


In [ ]:
# @title 7. Base Model Evaluation (Before Training)

FastModel.for_inference(model)

eval_prompt = "Patient on Warfarin 5mg and Omeprazole 20mg. What are the interactions?"
prompt = f"{system_prompt}\n<bos><start_of_turn>user\n{eval_prompt}<end_of_turn>\n<start_of_turn>model\n<unused0>\n"

inputs = tokenizer(
    text=[prompt],
    return_tensors="pt",
    add_special_tokens=False
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True)
base_response = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

print("========== BASE MODEL RESPONSE ==========")
print(base_response)
print("=========================================")

========== BASE MODEL RESPONSE ==========
You are a clinical deliberation AI. You must rigidly format your output exactly according to the following schema. Do not deviate or add conversational filler.

EXPECTED OUTPUT SCHEMA:
<unused0>
<unused1>
<unused5>
[Numbered step-by-step logical chain]

<unused6>
[Enumerate competing interpretations with probability estimates, including supporting and weakening evidence]

<unused7>
<unused9> Discarded: [Explanation of discarded paths]

<unused8> Selected: [Final chosen interpretation]
<unused3> [Optional tool calls to verify claims]
<unused2>
[🔴, 🟡, or 🟢] [Clinical reasoning summary]

Confidence: ✅ [LEVEL]

Recommendation: [Actionable advice]
<bos><start_of_turn>user
Patient on Warfarin 5mg and Omeprazole 20mg. What are the interactions?<end_of_turn>
<start_of_turn>model
<unused0>
    - Warfarin (5mg)
    - Omeprazole (20mg)

    - Omeprazole (20%)
    - Warfarin (5mg)

    - Omeprazole (20%)
    - Warfarin (5mg)

    - Omeprazole (20%)
    - W

In [ ]:
# @title 8. Configure Training with Clean Stack

from trl import SFTTrainer, SFTConfig

print("Configuring training...")

is_bfloat16 = torch.cuda.is_bf16_supported()

# Configure the Trainer (Standard 8-bit AdamW, no LoRA+, no NEFTune)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer,
    train_dataset=deliberation_dataset,
    dataset_text_field="text",
    max_seq_length=8192,
    args=SFTConfig(
        per_device_train_batch_size = 8, # Push this up until you OOM 2 earlier
        gradient_accumulation_steps = 1, # Push this down proportionally 3 earlier
        warmup_steps=50,
        num_train_epochs=4,
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        optim="adamw_8bit", # Standard 8-bit AdamW
        weight_decay=0.01,
        fp16=not is_bfloat16,
        bf16=is_bfloat16,
        logging_steps=5,
        save_steps=100,
        output_dir=os.path.join(project_dir, "outputs"),
        seed=3407,
    ),
)

print("✅ Training configured with Clean Stack")
print(f"✅ Optimizer: adamw_8bit")
print(f"✅ Scheduler: cosine")

Configuring training...


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/1011 [00:00<?, ? examples/s]

✅ Training configured with Clean Stack
✅ Optimizer: adamw_8bit
✅ Scheduler: cosine


In [ ]:
# @title 9. Train Model

print("Starting training...")

trainer.train()

print("✅ Training completed")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting training...


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,011 | Num Epochs = 4 | Total steps = 508
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 2,759,914,496 of 27,827,650,864 (9.92% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,5.594312
10,5.001956
15,3.709817
20,2.543572
25,1.932264
30,1.241666
35,0.769201
40,0.554136
45,0.451188
50,0.426536


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PRISM_FineTuning/outputs/checkpoint-100/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PRISM_FineTuning/outputs/checkpoint-200/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PRISM_FineTuning/outputs/checkpoint-300/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PRISM_FineTuning/outputs/checkpoint-400/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PRISM_FineTuning/outputs/checkpoint-500/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PRISM_FineTuning/outputs/checkpoint-508/tokenizer_config.json.


✅ Training completed


In [ ]:
# @title 10. Fine-Tuned Model Evaluation (After Training)

FastModel.for_inference(model)

inputs = tokenizer(
    text=[prompt],
    return_tensors="pt",
    add_special_tokens=False
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True)
tuned_response = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

print("========== FINE-TUNED MODEL RESPONSE ==========")
print(tuned_response)
print("=========================================")

def evaluate_structure(response):
    metrics = {
        "Has Logical Chain": "<unused5>" in response,
        "Has Competing Hypotheses": "<unused6>" in response,
        "Has Discarded Paths": "<unused7>" in response,
        "Has Tool Call": "<unused3>" in response,
        "Has Final Output Signal": any(dot in response for dot in ["🔴", "🟡", "🟢"]),
        "Has Confidence Badge": "Confidence:" in response
    }
    print("\n--- Structured Output Evaluation ---")
    score = sum(1 for passed in metrics.values() if passed)
    for key, passed in metrics.items():
        print(f"{key}: {'✅ Pass' if passed else '❌ Fail'}")
    print(f"\nOverall Structure Score: {score}/{len(metrics)}")
    return score

print("\nEvaluating Fine-Tuned Model:")
evaluate_structure(tuned_response)

========== FINE-TUNED MODEL RESPONSE ==========
You are a clinical deliberation AI. You must rigidly format your output exactly according to the following schema. Do not deviate or add conversational filler.

EXPECTED OUTPUT SCHEMA:
<unused0>
<unused1>
<unused5>
[Numbered step-by-step logical chain]

<unused6>
[Enumerate competing interpretations with probability estimates, including supporting and weakening evidence]

<unused7>
<unused9> Discarded: [Explanation of discarded paths]

<unused8> Selected: [Final chosen interpretation]
<unused3> [Optional tool calls to verify claims]
<unused2>
[🔴, 🟡, or 🟢] [Clinical reasoning summary]

Confidence: ✅ [LEVEL]

Recommendation: [Actionable advice]
<bos><start_of_turn>user
Patient on Warfarin 5mg and Omeprazole 20mg. What are the interactions?<end_of_turn>
<start_of_turn>model
<unused0>
<unused1>
<unused5>
1. Target drugs: Warfarin (Vitamin K antagonist) and Omeprazole (PPI).
2. Pharmacokinetics: Omeprazole inhibits CYP2C9 (minor pathway for S-

6

In [ ]:
import os

# Manually clone and compile llama.cpp to bypass Unsloth's internet check in Colab
if not os.path.exists("/root/.unsloth/llama.cpp"):
    !mkdir -p /root/.unsloth
    !git clone https://github.com/ggerganov/llama.cpp /root/.unsloth/llama.cpp

# Compile it using CMake since Makefile is deprecated
if not os.path.exists("/root/.unsloth/llama.cpp/llama-quantize"):
    !cd /root/.unsloth/llama.cpp && cmake -B build && cmake --build build --config Release -j
    # Unsloth checks the root directory for the binary, so we copy it over
    !cp /root/.unsloth/llama.cpp/build/bin/llama-quantize /root/.unsloth/llama.cpp/llama-quantize || true

print("Fusing weights and compiling BF16 GGUF directly from memory...")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

# Unsloth will automatically merge the LoRA adapters into the base model and export the GGUF
model.push_to_hub_gguf(
    "chandan989/prism-gemma-4-26B-A4B-it-bf16-v3.6",
    tokenizer,
    quantization_method = "bf16",
    token = hf_token
)

print("Process Complete. The GGUF has been successfully pushed to Hugging Face.")

Fusing weights and compiling BF16 GGUF directly from memory...
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /root/.cache/modelscope/hub/models/unsloth/gemma-4-26b-a4b-it


[transformers] Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_rtnbk7ks/tokenizer_config.json.


No existing and accessible Hugging Face cache directory found.


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [03:44<03:44, 224.50s/it]

Copied model-00001-of-00002.safetensors from local model directory


...

Uploading gemma-4-26b-a4b-it.BF16-mmproj.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b-a4b-it.BF16-mmproj.gguf:  17%|#7        |  206MB / 1.19GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/chandan989/prism-gemma-4-26B-A4B-it-bf16-v2.5
Unsloth: Cleaning up temporary files...
Process Complete. The GGUF has been successfully pushed to Hugging Face.


In [11]:
# @title 12. Compile llama.cpp for MXFP4

import os

if not os.path.exists("/root/.unsloth/llama.cpp"):
    !mkdir -p /root/.unsloth
    !git clone https://github.com/ggerganov/llama.cpp /root/.unsloth/llama.cpp

if not os.path.exists("/root/.unsloth/llama.cpp/build/bin/llama-quantize"):
    !cd /root/.unsloth/llama.cpp && cmake -B build -DGGML_CUDA=on && cmake --build build --config Release -j
    !cp /root/.unsloth/llama.cpp/build/bin/llama-quantize /root/.unsloth/llama.cpp/llama-quantize || true

print("✅ llama.cpp compiled with CUDA support")

Cloning into '/root/.unsloth/llama.cpp'...
remote: Enumerating objects: 92550, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 92550 (delta 198), reused 113 (delta 113), pack-reused 92246 (from 3)
Receiving objects: 100% (92550/92550), 384.83 MiB | 48.73 MiB/s, done.
Resolving deltas: 100% (65601/65601), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification 

In [12]:
# @title 13. Calibrate I-Matrix and Quantize to MXFP4

from huggingface_hub import snapshot_download
import glob
import os

hf_token = userdata.get("HF_TOKEN")

print("Downloading BF16 model from Hugging Face...")
snapshot_download(
    repo_id="chandan989/prism-gemma-4-26B-A4B-it-bf16-v3.6",
    allow_patterns=["*.gguf"],
    local_dir=".",
    token=hf_token
)

# Find the BF16 GGUF file
gguf_files = [f for f in glob.glob("*.gguf") if "bf16" in f.lower() and "mmproj" not in f.lower()]
gguf_files.sort()

if gguf_files:
    bf16_model = gguf_files[0]
    print(f"Found BF16 model: {bf16_model}")

    # 1. GENERATE CALIBRATION DATASET
    print("Generating raw text calibration file for I-Matrix...")
    calib_path = "prism_calibration.txt"
    with open(calib_path, "w", encoding="utf-8") as f:
      for item in deliberation_dataset:
        f.write(item["text"] + "\n\n")

    # 2. CALCULATE IMPORTANCE MATRIX (No hallucinated flags)
    print("Calculating Importance Matrix (This takes time)...")
    !/root/.unsloth/llama.cpp/build/bin/llama-imatrix -m {bf16_model} -f {calib_path} -o imatrix.dat


    # 3. QUANTIZE TO MXFP4 AND SHIELD EMBEDDINGS
    print("Running MXFP4 quantization with I-Matrix and Embedding Shielding...")
    !/root/.unsloth/llama.cpp/build/bin/llama-quantize --imatrix imatrix.dat --leave-output-tensor --token-embedding-type f16 {bf16_model} prism-mxfp4.gguf MXFP4_MOE
    print("✅ MXFP4 quantization complete")
    print("✅ Output tensors and embeddings locked at F16 to prevent amnesia")
else:
    print("❌ BF16 GGUF not found")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
save_imatrix: entry '           blk.4.ffn_gate_up_exps.weight' has partial data (95.31%)
save_imatrix: entry '          blk.24.ffn_gate_up_exps.weight' has partial data (98.44%)
save_imatrix: entry '           blk.5.ffn_gate_up_exps.weight' has partial data (97.66%)

[49]6.9207,[50]6.8300,[51]6.9371,[52]6.8897,[53]6.9577,[54]6.9657,[55]6.9548,[56]6.9957,
save_imatrix: saving imatrix using GGUF format with a different suffix than .gguf
save_imatrix: if you want the previous imatrix format, use --output-format dat

save_imatrix: entry '          blk.29.ffn_gate_up_exps.weight' has partial data (96.88%)
save_imatrix: entry '             blk.15.ffn_down_exps.weight' has partial data (98.44%)
save_imatrix: entry '             blk.20.ffn_down_exps.weight' has partial data (97.66%)
save_imatrix: entry '              blk.9.ffn_down_exps.weight' has partial data (99.22%)
save_imatrix: entry '           blk.3.ffn_gate_up_exps.weight' has partial

In [13]:
# @title 14. Upload MXFP4 to Hugging Face

from huggingface_hub import HfApi

api = HfApi()
model_name = "prism-gemma-4-26B-A4B-it-MXFP4-v3.6"
repo_id = f"chandan989/{model_name}"
path = "prism-mxfp4.gguf"

if os.path.exists(path):
    print(f"Creating repository {repo_id}...")
    api.create_repo(repo_id=repo_id, exist_ok=True, token=hf_token)

    print("Uploading MXFP4 model with I-Matrix calibration...")
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=f"{model_name}.gguf",
        repo_id=repo_id,
        token=hf_token
    )
    print("✅ MXFP4 model uploaded")
    print("✅ Model includes I-Matrix protection for special tokens")
else:
    print("❌ MXFP4 file not found")

Creating repository chandan989/prism-gemma-4-26B-A4B-it-MXFP4-v3.6...
Uploading MXFP4 model with I-Matrix calibration...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  prism-mxfp4.gguf            :   0%|          | 1.18MB / 15.4GB            

✅ MXFP4 model uploaded
✅ Model includes I-Matrix protection for special tokens


In [ ]:
# @title 15. Upload to Kaggle

!pip install -q kaggle

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

kaggle_dir = "kaggle_model_upload_dir"
os.makedirs(kaggle_dir, exist_ok=True)

# Symlink the MXFP4 file
if os.path.exists(path):
    dest_path = os.path.join(kaggle_dir, os.path.basename(path))
    if not os.path.exists(dest_path):
        os.symlink(os.path.abspath(path), dest_path)

    kaggle_username = os.environ['KAGGLE_USERNAME']
    model_slug = "prism-gemma-4-26b-a4b-it"
    framework = "gguf"
    instance_slug = "mxfp4-v4-clean"

    print(f"Setting up Kaggle Model: {kaggle_username}/{model_slug}...")
    !kaggle models create {kaggle_username}/{model_slug} || true
    !kaggle models instances create {kaggle_username}/{model_slug}/{framework}/{instance_slug} || true

    print(f"Uploading to Kaggle...")
    !kaggle models instances versions create {kaggle_username}/{model_slug}/{framework}/{instance_slug} --dir {kaggle_dir}

    print(f"✅ Uploaded to Kaggle: https://www.kaggle.com/models/{kaggle_username}/{model_slug}")
    print("✅ Model includes I-Matrix calibration for special token protection")
else:
    print("❌ MXFP4 file not found")

In [ ]:
# @title 16. Test MXFP4 Model with llama.cpp

# Install llama-cpp-python with CUDA support for hardware acceleration
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python

from huggingface_hub import login, snapshot_download
import llama_cpp
import time
import glob

# Download MXFP4 model
# repo_id = "chandan989/prism-gemma-4-26B-A4B-it-bf16-v3.5"
repo_id = "chandan989/prism-gemma-4-26B-A4B-it-MXFP4-v3.6"
print(f"Downloading {repo_id}...")
model_path = snapshot_download(repo_id=repo_id, local_dir="/content/model")

# Find GGUF file
gguf_files = glob.glob(f"{model_path}/*.gguf")
gguf_files.sort()
MODEL_PATH = gguf_files[0] if gguf_files else None

print(f"Loading MXFP4 model: {MODEL_PATH}")
print("✅ Model includes I-Matrix calibration for special token protection")

# Load model with full context window
start = time.time()
llm = llama_cpp.Llama(
    model_path=MODEL_PATH,
    n_ctx=8192,  # Full context window
    n_gpu_layers=32,  # More layers for better performance
    type_k=llama_cpp.GGML_TYPE_Q8_0,
    type_v=llama_cpp.GGML_TYPE_Q8_0,
    flash_attn=True,
    n_threads=4,
    verbose=False,
)
elapsed = time.time() - start
print(f"✅ Model loaded in {elapsed:.1f}s")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 MB 40.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
# @title 17. Run Holdout Tests on MXFP4 Model

# Strict system prompt
system_prompt = """You are a clinical deliberation AI. You must rigidly format your output exactly according to the following schema. Do not deviate or add conversational filler.

EXPECTED OUTPUT SCHEMA:
<unused0>
<unused1>
<unused5>
[Numbered step-by-step logical chain]

<unused6>
[Enumerate competing interpretations with probability estimates, including supporting and weakening evidence]

<unused7>
<unused9> Discarded: [Explanation of discarded paths]

<unused8> Selected: [Final chosen interpretation]
<unused3> [Optional tool calls to verify claims]
<unused2>
[🔴, 🟡, or 🟢] [Clinical reasoning summary]

Confidence: ✅ [LEVEL]

Recommendation: [Actionable advice]"""

def run_inference(eval_prompt, max_tokens=3072):
    prompt_str = f"{system_prompt}\n<bos><start_of_turn>user\n{eval_prompt}<end_of_turn>\n<start_of_turn>model\n<unused0>\n"
    prompt_tokens = llm.tokenize(prompt_str.encode("utf-8"), add_bos=False, special=True)

    output = llm(
        prompt_tokens,
        max_tokens=max_tokens,
        temperature=0.3,
        top_p=0.85,
        stop=["<eos>", "<end_of_turn>"],
        echo=False
    )
    return output['choices'][0]['text']

def evaluate_structure(response):
    # We check for text markers since llama.cpp strips <unusedX> tokens by default
    # metrics = {
    #     "has_discarded_paths": "Discarded:" in response,
    #     "has_selected_path": "Selected:" in response,
    #     "has_signal_dot": any(dot in response for dot in ["🔴", "🟡", "🟢"]),
    #     "has_confidence_badge": "Confidence: ✅" in response,
    #     "has_recommendation": "Recommendation:" in response
    # }
    metrics = {
    "has_discarded_paths": "<unused7>" in response,
    "has_selected_path": "<unused8>" in response,
    "has_signal_dot": any(dot in response for dot in ["🔴", "🟡", "🟢"]),
    "has_confidence_badge": "Confidence: ✅" in response,
    "has_recommendation": "Recommendation:" in response
}
    score = sum(1 for passed in metrics.values() if passed)
    return metrics, score, len(metrics)

def evaluate_accuracy(response, expected_dot):
    return expected_dot in response

def print_eval(metrics, score, total, is_accurate, expected):
    print(f"Structure Score: {score}/{total}")
    for k, v in metrics.items():
        if k not in ["score", "total", "query", "is_accurate"]:
            print(f"  {k}: {'✅' if v else '❌'}")
    print(f"Accuracy (matches expected {expected}): {'✅' if is_accurate else '❌'}")

# Holdout test suite
HOLDOUT_TESTS = [
    {"prompt": "Patient on Simvastatin 80mg just started Itraconazole 200mg for toenail fungus. Evaluate.", "expected_dot": "🔴"},
    {"prompt": "72yo female on Digoxin 0.25mg and Amiodarone 200mg. Review all interactions.", "expected_dot": "🟡"},
    {"prompt": "Patient on Metformin 500mg and Atorvastatin 20mg. Any interaction?", "expected_dot": "🟢"},
    {"prompt": "82yo nursing home resident on: Warfarin 5mg, Omeprazole 40mg, Clopidogrel 75mg, Aspirin 81mg, Metoprolol 50mg, Sertraline 100mg, Gabapentin 300mg, Amlodipine 10mg, Metformin 1000mg, Glipizide 10mg. Identify the three most dangerous interactions.", "expected_dot": "🔴"},
    # Additional tests for safe combinations
    {"prompt": "Patient on Lisinopril 10mg and Hydrochlorothiazide 25mg. Any interaction?", "expected_dot": "🟢"},
    {"prompt": "Patient on Levothyroxine 100mcg and Omeprazole 20mg. Any interaction?", "expected_dot": "🟢"},
]

print(f"Running {len(HOLDOUT_TESTS)} holdout tests on MXFP4 model...")
print("✅ Model includes I-Matrix calibration for special token protection")
print()
all_metrics = []

for i, test in enumerate(HOLDOUT_TESTS, 1):
    prompt = test["prompt"]
    expected = test["expected_dot"]

    print(f"{'='*60}")
    print(f"TEST {i}/{len(HOLDOUT_TESTS)}")
    print(f"QUERY: {prompt[:100]}...")
    print(f"EXPECTED: {expected}")
    print(f"{'='*60}")

    response = run_inference(prompt)
    metrics, score, total = evaluate_structure(response)
    is_accurate = evaluate_accuracy(response, expected)

    # Store for summary
    metrics["score"] = score
    metrics["total"] = total
    metrics["is_accurate"] = is_accurate
    metrics["query"] = prompt[:80]
    all_metrics.append(metrics)

    print_eval(metrics, score, total, is_accurate, expected)
    print()

In [ ]:
# @title 18. Generate Compliance Summary

import statistics

fields = [
    "has_discarded_paths", "has_selected_path", "has_signal_dot",
    "has_confidence_badge", "has_recommendation"
]

print("=" * 60)
print("P.R.I.S.M. STRUCTURAL COMPLIANCE SUMMARY (MXFP4)")
print("=" * 60)
print("✅ Model includes I-Matrix calibration for special token protection")
print()

for field in fields:
    hits = sum(1 for m in all_metrics if m.get(field, False))
    pct = (hits / len(all_metrics)) * 100 if all_metrics else 0
    bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    label = field.replace("has_", "").replace("_", " ").title()
    print(f"{label:<25} {bar} {hits}/{len(all_metrics)} ({pct:.0f}%)")

scores = [m["score"] for m in all_metrics]
avg = statistics.mean(scores) if scores else 0
total = all_metrics[0]["total"] if all_metrics else 1
overall = (avg / total) * 100

pass_threshold = int(total * 0.75)

accurate_hits = sum(1 for m in all_metrics if m.get("is_accurate", False))
accuracy_pct = (accurate_hits / len(all_metrics)) * 100 if all_metrics else 0

print(f"\n{'─' * 60}")
print(f"  Overall Structural Compliance: {overall:.1f}% ({avg:.1f}/{total} avg)")
print(f"  Tests Passed Structure (≥{pass_threshold}/{total}): {sum(1 for s in scores if s >= pass_threshold)}/{len(scores)}")
print(f"  Overall Clinical Accuracy:     {accuracy_pct:.1f}% ({accurate_hits}/{len(all_metrics)})")
print(f"{'─' * 60}")

# Compare with Experiment 1
print("\n" + "=" * 60)
print("COMPARISON WITH EXPERIMENT 1")
print("=" * 60)
print(f"Experiment 1 (Previous): 75.0% (3.8/5 avg)")
print(f"Experiment 2 (MXFP4): {overall:.1f}% ({avg:.1f}/{total} avg)")
print(f"Improvement: {overall - 75.0:+.1f}%")
print()
print("Key Improvements in Experiment 2 (The Clean Stack):")
print("  ✅ Removed DoRA to restore Triton MoE coalescing")
print("  ✅ Removed LoRA+ and NEFTune for exact structural memorization")
print("  ✅ rsLoRA for rank stabilization at r=64")
print("  ✅ Standard 8-bit AdamW optimizer")
print("  ✅ I-Matrix calibration to protect special tokens")
print("  ✅ MXFP4 quantization with importance matrix")
print("  ✅ Fixed Evaluation: Isolated structural parsing vs accuracy")

In [ ]:
# @title 19. Display Glass Box View

import re

def display_glass_box(response: str):
    """Parse and pretty-print a P.R.I.S.M. response."""
    thought = ""
    answer = ""

    parts = response.split("<unused2>")

    if len(parts) >= 2:
        thought_part = parts[0]
        if "<unused1>" in thought_part:
            thought = thought_part.split("<unused1>")[-1].strip()
        else:
            thought = thought_part.replace("<unused0>", "").strip()
        answer = parts[1].strip()
    else:
        match = re.search(r'(🟢|🟡|🔴)', response)
        if match:
            split_idx = match.start()
            thought = response[:split_idx].replace("<unused0>", "").replace("<unused1>", "").strip()
            answer = response[split_idx:].strip()
        else:
            thought = response.replace("<unused0>", "").replace("<unused1>", "").strip()
            answer = "(Model failed to utilize channel tags or signal dots)"

    # Restore human-readable tags
    thought = thought.replace("<unused5>", "[Logical Chain]").replace("<unused6>", "[Competing Hypotheses]").replace("<unused7>", "[Discarded Paths]").replace("<unused8>", "▶ Selected:")
    answer = answer.replace("<unused3>", "<|tool_call>").replace("<unused4>", '<"|>')

    print("┌─── 🧠 DELIBERATION TRACE (Expert View) ─────────────┐")
    for line in thought.split("\n"):
        if line.strip():
            print(f"│  {line.strip()}")
    print("└─────────────────────────────────────────────────────┘\n")

    print("┌─── 📋 CLINICAL ANSWER (Default View) ───────────────┐")
    for line in answer.split("\n"):
        if line.strip():
            print(f"│  {line.strip()}")
    print("└─────────────────────────────────────────────────────┘")

# Display the last test response
if all_metrics:
    last_response = run_inference(HOLDOUT_TESTS[-1]["prompt"])
    display_glass_box(last_response)